# ITCC508 Lab Exercise PT-M1
# Building and Evaluating a Domain-Specific RAG Chatbot

**Author:** Mark Emmanuel P. Sagarino  |  **Course:** ITC-C508 — Introduction to LLMOps and RAG Concepts

**Domain:** Household Solid-Waste Segregation and Recycling (Philippines)  
**SDG Alignment:** SDG 11 — Sustainable Cities and Communities · SDG 12 — Responsible Consumption and Production

---

### What this notebook builds
This notebook implements a **Retrieval-Augmented Generation (RAG)** chatbot that answers questions about
household waste segregation and recycling using **only** a small private corpus of documents that I supply. A
plain large language model answers from its training memory and will confidently invent details it does not
know — this is called **hallucination**. RAG reduces that risk by retrieving relevant passages from my own
documents first, then instructing the model to answer strictly from those passages.

**Pipeline:** Document Loading → Chunking → `all-MiniLM-L6-v2` Embeddings → ChromaDB Vector Store →
Retriever (top-k) → Grounded Prompt → Groq LLM

The lab template names `llama-3.1-8b-instant`, but Groq has since deprecated that model. Cell 4B
therefore selects an available model at run time; on this run it resolved to `openai/gpt-oss-20b`.

**Requirement note:** every code cell below is preceded by a Markdown cell that explains, line by line, what the
code does and why that step is necessary to the RAG architecture.

**Before you run:** you need a free Groq API key from `console.groq.com`. The key is requested at runtime and is
never written into the notebook.

## Cell 1 — Environment Setup and Package Installation

This first cell installs every third-party library the pipeline depends on. Colab starts from a clean machine, so
nothing here is pre-installed.

**Line-by-line**
1. `!pip install -q` — the leading `!` runs a shell command from inside the notebook; `-q` keeps the output quiet.
2. `langchain` — the orchestration framework that connects the retriever, the prompt, and the model into one chain.
3. `langchain-community` — community-maintained integrations; this is where `DirectoryLoader` and the `Chroma`
   vector-store wrapper live.
4. `langchain-groq` — the connector that lets LangChain call models hosted on Groq.
5. `langchain-huggingface` — provides `HuggingFaceEmbeddings`, the wrapper around the sentence-transformer model
   that converts text into vectors.
6. `chromadb` — the vector database itself, which stores the embeddings and performs similarity search.
7. `pypdf` and `unstructured` — file parsers, so the loader can read PDF and plain-text documents.
8. `sentence-transformers` — the engine that actually runs `all-MiniLM-L6-v2`; `HuggingFaceEmbeddings` calls it
   internally, so it must be present.

**Why this matters:** without these packages there is no loader to read the documents, no embedding model to
vectorise them, no database to search them, and no client to reach the LLM.

**Debugging note — why the versions are pinned below 1.0.** On the first run, installing the packages without a
version constraint pulled in **LangChain 1.x**, and Cell 6 then failed with
`ModuleNotFoundError: No module named 'langchain.chains'`. LangChain 1.0 restructured the main package and moved
the legacy chain builders — `create_retrieval_chain` and `create_stuff_documents_chain`, both of which the lab
template uses — out of `langchain.chains`. Pinning every LangChain package to `<1.0` keeps the 0.3-generation API
that the template was written against, so the provided code runs unmodified. This was the one genuine
dependency problem encountered in this exercise, and it is recorded in the report's references.

In [ ]:
# NOTE: the langchain packages are pinned below version 1.0 on purpose - see the markdown above.
!pip install -q "langchain<1.0" "langchain-core<1.0" "langchain-community<1.0" \
                "langchain-groq<1.0" "langchain-huggingface<1.0" "langchain-text-splitters<1.0" \
                chromadb pypdf unstructured sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.9 MB/s eta 0:00:00
   ━

## Cell 2 — Secure API Key Configuration

The Groq API key is a secret credential. Anyone holding it can spend the quota attached to my account, so it must
never be typed as plain text into a notebook that will be submitted or shared.

**Line-by-line**
1. `import os` — gives access to `os.environ`, the process environment where libraries look for credentials.
2. `import getpass` — provides a masked input prompt; characters typed are not echoed to the screen.
3. `if "GROQ_API_KEY" not in os.environ:` — only prompts when the key is not already set, so re-running the cell
   does not ask again unnecessarily.
4. `os.environ["GROQ_API_KEY"] = getpass.getpass(...)` — stores the key in memory for this session only.
   `ChatGroq` reads it automatically from this variable, so the key never needs to appear in any later cell.

**Why this matters (security practice):** the key exists only in the runtime's memory. It is not saved to disk, not
printed in any output, and it disappears when the runtime is recycled. Hard-coding a key as a string literal would
permanently embed the secret in the submitted `.ipynb` file.

In [ ]:
import os
import getpass

# Prompts for the API key securely, without echoing or storing plain text in the notebook
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

print("Groq API key configured for this session.")

Enter your Groq API Key: ··········
Groq API key configured for this session.


## Cell 2B — Creating `./my_data/` and Uploading the Domain Corpus

The exercise requires the source documents to live in a folder named `./my_data/`. This cell creates that folder
and lets me upload my three `.txt` files into it.

**Line-by-line**
1. `os.makedirs("my_data", exist_ok=True)` — creates the directory; `exist_ok=True` prevents an error if it is
   already there, which makes the cell safe to re-run.
2. `from google.colab import files` — imports Colab's file-transfer helper.
3. `files.upload()` — opens a file picker and returns a dictionary of the uploaded filenames.
4. `shutil.move(...)` — Colab drops uploads into the working directory, so each file is moved into `my_data/`.
5. The `for` loop deletes any filename containing `~`. Windows sometimes supplies a legacy 8.3 short name
   (for example `01_RA9~1.TXT`) in addition to the full filename, which would place a duplicate copy of the same
   document in the corpus and distort both the chunk count and the file listing reported in Task 1.
6. `sorted(os.listdir("my_data"))` — prints the final contents so I can confirm exactly three documents landed.

**Why this matters:** the loader in the next cell reads whatever it finds in this folder. If the folder is empty or
the files are in the wrong place, the vector store will be built from nothing and every answer will fall back to
"I cannot answer based on the provided domain data."

**Files to upload:** `01_RA9003_solid_waste_law.txt`, `02_city_waste_segregation_guide.txt`,
`03_recycling_circular_economy_faq.txt`

In [ ]:
import os, shutil

os.makedirs("my_data", exist_ok=True)

# Upload the three domain .txt files when prompted
try:
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if os.path.exists(fname):
            shutil.move(fname, os.path.join("my_data", fname))
except ImportError:
    print("Not running in Colab - place the .txt files inside ./my_data/ manually.")

# Remove Windows 8.3 short-name duplicates (e.g. 01_RA9~1.TXT) if any were uploaded alongside the long names
for f in list(os.listdir("my_data")):
    if "~" in f:
        os.remove(os.path.join("my_data", f))
        print("Removed duplicate short-name file:", f)

print("Contents of ./my_data/:", sorted(os.listdir("my_data")))

Saving 01_RA9003_solid_waste_law.txt to 01_RA9003_solid_waste_law.txt
Saving 02_city_waste_segregation_guide.txt to 02_city_waste_segregation_guide.txt
Saving 03_recycling_circular_economy_faq.txt to 03_recycling_circular_economy_faq.txt
Contents of ./my_data/: ['01_RA9003_solid_waste_law.txt', '02_city_waste_segregation_guide.txt', '03_recycling_circular_economy_faq.txt']


## Cell 3 — Loading Custom Data and Chunking

This cell turns raw files into the small text passages that the retriever will later search. It is the first true
stage of the RAG pipeline.

**Line-by-line**
1. `from langchain_community.document_loaders import DirectoryLoader, TextLoader` — `DirectoryLoader` walks a
   folder; `TextLoader` reads one plain-text file.
2. `from langchain_text_splitters import RecursiveCharacterTextSplitter` — the splitter that cuts long documents
   into chunks.
3. `DirectoryLoader("my_data/", glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
   show_progress=True)` — `glob="**/*.txt"` matches every `.txt` file in the folder and its sub-folders.
   **Debugging note:** the original template used `glob="**/*.*"`, which routes every file through
   `UnstructuredFileLoader` and frequently raises errors or missing-dependency warnings. Passing
   `loader_cls=TextLoader` with an explicit UTF-8 encoding makes the load deterministic and fast for a plain-text
   corpus.
4. `loader.load()` — returns a list of `Document` objects, one per file, each carrying `page_content` and a
   `metadata` dictionary that records the source filename.
5. `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` — configures the splitter.
   `chunk_size=500` caps each chunk at roughly 500 characters. `chunk_overlap=50` repeats the last 50 characters
   of one chunk at the start of the next. The splitter is *recursive* because it first tries to break on
   paragraph breaks, then sentences, then words, so it cuts at natural boundaries rather than mid-word.
6. `text_splitter.split_documents(raw_documents)` — performs the split and preserves each chunk's source metadata.

**Why chunking is necessary:** embeddings represent a whole passage as a single vector, so a long document
compresses into one blurred point that matches nothing precisely. Small chunks keep each vector focused on one
idea, which sharply improves retrieval accuracy. The 50-character overlap exists so that a sentence split across
a chunk boundary is not lost — its context still appears intact in one of the two neighbouring chunks.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.makedirs("my_data", exist_ok=True)

# Load every .txt document from the folder.
# TextLoader is specified explicitly (instead of the default Unstructured loader) for reliability.
loader = DirectoryLoader(
    "my_data/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
raw_documents = loader.load()

# Split documents into smaller semantic chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")
print("\nSource files:")
for d in raw_documents:
    print(" -", d.metadata.get("source"))
print("\nExample chunk:\n", documents[0].page_content[:300])

100%|██████████| 3/3 [00:00<00:00, 2022.33it/s]

Loaded 3 raw document(s) and split into 38 chunks.

Source files:
 - my_data/01_RA9003_solid_waste_law.txt
 - my_data/02_city_waste_segregation_guide.txt
 - my_data/03_recycling_circular_economy_faq.txt

Example chunk:
 REPUBLIC ACT 9003 - ECOLOGICAL SOLID WASTE MANAGEMENT ACT OF 2000
A Plain-Language Summary for Residents and Barangay Officials
(Prepared as reference material for a community information chatbot. SDG 11 and SDG 12.)


## Cell 4 — Embedding Model and Vector Database Indexing

This cell converts every text chunk into a numerical vector and stores those vectors in a searchable database.
This is the "retrieval" half of Retrieval-Augmented Generation.

**Line-by-line**
1. `from langchain_huggingface import HuggingFaceEmbeddings` — the wrapper for open-source sentence-transformer
   models.
2. `from langchain_community.vectorstores import Chroma` — the LangChain interface to the ChromaDB vector store.
3. `HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` — downloads and loads the embedding model.
   `all-MiniLM-L6-v2` maps a passage of text into a **384-dimensional** vector positioned so that passages with
   similar meaning sit close together. It is chosen here because it is small, fast, free, and runs locally on CPU,
   so no text is sent to an external embedding service.
4. `chromadb.EphemeralClient()` + `client.delete_collection("ptm1_waste")` — creates an explicit in-memory
   client and drops any collection left behind by an earlier execution **in the same session**. This guard
   is deliberate. `Chroma.from_documents()` writes into a collection that already exists rather than
   replacing it, so running the indexing cell twice without restarting the runtime loads a second copy of
   every chunk. The store then holds 76 vectors instead of 38, and a top-3 search returns the same passage
   twice plus one other — the effective `k` silently drops from 3 to about 2, and a passage that should
   have been retrieved is pushed out. Nothing errors; the answers just quietly get worse.
5. `Chroma.from_documents(...)` — embeds every chunk and builds the **in-memory** index. Nothing is written
   to disk, so the database is rebuilt each session.
6. `vectorstore.as_retriever(search_kwargs={"k": 3})` — wraps the store as a retriever. `k=3` means each query
   returns the **three** most similar chunks.

**Why this matters:** the retriever is what keeps the chatbot grounded. When a question arrives it is embedded
with the same model, and Chroma returns the chunks whose vectors are nearest by cosine similarity. Those chunks —
and nothing else — become the context handed to the LLM.

**Why `k=3`:** too small a `k` risks missing the passage that holds the answer; too large a `k` floods the prompt
with irrelevant text, which both raises cost and gives the model more room to drift off-source. Three is a
balanced default for a corpus of this size.

**Integrity check.** The cell prints the number of chunks handed to the index alongside the number of
vectors Chroma actually holds, and states plainly whether they match. A duplicated index is invisible in the
answers themselves, so it has to be made visible here, at the point where it would be introduced.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb

# Initialise the open-source embedding model (384-dimensional, runs locally)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Use an explicit client and a named collection, and drop any collection left over
# from an earlier execution in this same session. Without this guard, re-running the
# cell appends a SECOND copy of every chunk to the same default collection, so the
# top-k retrieval starts returning duplicates instead of k distinct passages. That
# failure is silent, which is exactly what makes it dangerous.
client = chromadb.EphemeralClient()
try:
    client.delete_collection("ptm1_waste")
except Exception:
    pass

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    client=client,
    collection_name="ptm1_waste",
)

# Expose the vector store as a retriever that returns the top-3 most similar chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

n_in  = len(documents)
n_idx = vectorstore._collection.count()
print(f"Chunks handed to the index : {n_in}")
print(f"Vectors actually in Chroma : {n_idx}")
print("INTEGRITY CHECK          :", "OK - no duplicates" if n_in == n_idx
      else f"FAILED - {n_idx - n_in} duplicate vector(s) present")
print("Retriever configured with k = 3")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks handed to the index : 38
Vectors actually in Chroma : 38
INTEGRITY CHECK          : OK - no duplicates
Retriever configured with k = 3


## Cell 4B — Model Availability Check (Handling a Deprecated Model)

**Why this cell exists.** The lab template specifies `llama-3.1-8b-instant`. On the first run this returned
`404 model_not_found`. Groq announced the deprecation of `llama-3.1-8b-instant` on 17 June 2026 and shut it down
for free and developer-tier traffic on 16 August 2026, recommending `openai/gpt-oss-20b` as the replacement. This
is a real LLMOps failure mode: a hosted model that a pipeline depends on can be retired underneath it, so the
code must not hard-code a single model identifier without a fallback.

**Line-by-line**
1. `import requests` — the HTTP client used to query Groq's REST API directly.
2. `requests.get("https://api.groq.com/openai/v1/models", headers={...})` — calls the models endpoint,
   authenticating with the key already stored in the environment. The response lists exactly the models this API
   key is entitled to use.
3. `available = sorted(...)` — extracts each model's `id` into a sorted list.
4. `preferred = [...]` — an ordered preference list. The template's original model is tried first so that the
   notebook still honours the specification if the model is ever restored, followed by the officially
   recommended replacement, then larger alternatives.
5. `next((m for m in preferred if m in available), available[0])` — selects the first preferred model that is
   actually available; if none of them are, it falls back to whatever the account does offer.
6. `MODEL_NAME` is then used by every later cell instead of a hard-coded string.

**Why this matters:** this turns a hard failure into a graceful degradation. It also documents the substitution
honestly, which is what the report must reflect — the architecture is unchanged, only the generator model differs
from the one named in the template.

In [ ]:
import requests

# Ask Groq which models this API key can actually use
resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)
available = sorted(m["id"] for m in resp.json().get("data", []))

print("Models available to this API key:")
for m in available:
    print("  -", m)

# The lab template specifies llama-3.1-8b-instant, which Groq deprecated (shut down 16 Aug 2026).
# Try it first for fidelity, then fall back to the officially recommended replacement.
preferred = [
    "llama-3.1-8b-instant",     # model named in the lab template
    "openai/gpt-oss-20b",       # Groq's recommended replacement
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
]
MODEL_NAME = next((m for m in preferred if m in available), available[0])

print()
print("=" * 60)
print("SELECTED MODEL:", MODEL_NAME)
if MODEL_NAME != "llama-3.1-8b-instant":
    print("NOTE: the template model 'llama-3.1-8b-instant' is unavailable (deprecated by Groq).")
    print("      This substitution is documented in the lab report.")
print("=" * 60)

Models available to this API key:
  - allam-2-7b
  - canopylabs/orpheus-arabic-saudi
  - canopylabs/orpheus-v1-english
  - groq/compound
  - groq/compound-mini
  - meta-llama/llama-prompt-guard-2-22m
  - meta-llama/llama-prompt-guard-2-86m
  - openai/gpt-oss-120b
  - openai/gpt-oss-20b
  - openai/gpt-oss-safeguard-20b
  - qwen/qwen3.6-27b
  - qwen/qwen3.8-27b
  - whisper-large-v3
  - whisper-large-v3-turbo

SELECTED MODEL: openai/gpt-oss-20b
NOTE: the template model 'llama-3.1-8b-instant' is unavailable (deprecated by Groq).
      This substitution is documented in the lab report.


## Cell 5 — Model Initialisation and the Grounded Domain System Prompt

This cell creates the language model and, more importantly, writes the instruction that constrains it. The system
prompt is the main software control against hallucination.

**Line-by-line**
1. `from langchain_groq import ChatGroq` — the client for models served by Groq.
2. `from langchain_core.prompts import ChatPromptTemplate` — builds a reusable prompt with placeholders.
3. `ChatGroq(model_name=MODEL_NAME, temperature=0)` — instantiates the model, using the identifier resolved
   in Cell 4B rather than a hard-coded name.
   **`temperature=0`** makes generation effectively deterministic: at each step the model takes the highest-probability
   token instead of sampling. For a factual assistant this is what we want — the same question should return the
   same answer, and the model should not get "creative".
4. `system_prompt = (...)` — the instruction block. Three rules do the work:
   - it declares a narrow role, a specialist for this domain only;
   - **"Answer questions strictly using ONLY the provided context below"** forbids the model from drawing on its
     own pre-trained knowledge;
   - **the fallback rule** gives it a safe, explicit way to say "I don't know" instead of guessing.
5. `"Context:\n{context}"` — `{context}` is a placeholder that LangChain fills at run time with the retrieved
   chunks.
6. `ChatPromptTemplate.from_messages([...])` — assembles the final template: the `system` turn carries the rules
   and the retrieved context, and the `human` turn carries `{input}`, the user's actual question.

**Why this matters:** retrieval supplies the facts, but nothing forces the model to *use* them. The system prompt
supplies that obligation. Together, `temperature=0` (no random sampling) and the fallback rule (a permitted refusal)
form the baseline configuration that Task 3 will deliberately dismantle to measure hallucination.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Initialise the small language model, deterministic setting
llm = ChatGroq(
    model_name=MODEL_NAME,      # resolved in Cell 4B (template model is deprecated)
    temperature=0
)

# Custom domain system prompt - the grounding contract
system_prompt = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n"
    "If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

print(f"LLM initialised: {MODEL_NAME} | temperature = 0")
print("Grounded system prompt with fallback rule is active.")

LLM initialised: openai/gpt-oss-20b | temperature = 0
Grounded system prompt with fallback rule is active.


## Cell 6 — Pipeline Assembly

This cell wires the retriever, the prompt, and the model into a single callable chain.

**Line-by-line**
1. `from langchain.chains import create_retrieval_chain` — builds the outer chain that runs retrieval first.
2. `from langchain.chains.combine_documents import create_stuff_documents_chain` — builds the inner chain that
   places retrieved documents into the prompt.
3. `create_stuff_documents_chain(llm, prompt)` — the "stuff" strategy concatenates all retrieved chunks and stuffs
   them into the `{context}` placeholder in one call. It suits a small `k` such as 3, where everything fits
   comfortably inside the context window.
4. `create_retrieval_chain(retriever, combine_docs_chain)` — produces the complete RAG chain. On each call it
   embeds the question, retrieves the top-k chunks, fills `{context}`, sends the assembled prompt to the model,
   and returns a dictionary containing both the `answer` and the `context` chunks that produced it.

**Why this matters:** returning the retrieved `context` alongside the answer is what makes the system auditable.
Every claim can be traced back to the exact chunk and source file it came from, which is the practical difference
between a grounded assistant and an unverifiable one.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Combine the prompt and the LLM into a document-processing chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Assemble the full retrieval-augmented generation chain
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print(f"RAG chain assembled: retriever -> prompt -> {MODEL_NAME}")

RAG chain assembled: retriever -> prompt -> openai/gpt-oss-20b


## Cell 7 — Testing the Custom Domain Chatbot (In-Domain Queries)

This cell verifies that the pipeline answers correctly for questions whose answers genuinely exist in the corpus.

**Line-by-line**
1. `queries = [...]` — four in-domain questions, each targeting a different source document so retrieval is
   exercised across the whole corpus.
2. `rag_chain.invoke({"input": q})` — runs the full chain for one question. The dictionary key `input` matches the
   `{input}` placeholder defined in Cell 5.
3. `response["answer"]` — the generated text.
4. `response["context"]` — the list of chunks the retriever actually supplied.
5. The loop over `response["context"]` prints each chunk's `metadata["source"]`, showing which file the answer was
   grounded in.

**Why this matters:** printing the retrieved sources next to the answer is the verification step. If the answer is
correct but the sources are unrelated, the model is answering from memory rather than from the corpus — which
would mean the grounding has failed even though the output looks right.

In [ ]:
queries = [
    "What are the four waste categories and which colour container should each use?",
    "What is the penalty for open burning of solid waste under RA 9003?",
    "What is a Materials Recovery Facility and who is required to establish one?",
    "Are plastic sachets recyclable, and why or why not?",
]

for q in queries:
    response = rag_chain.invoke({"input": q})
    print("=" * 90)
    print("QUERY:", q)
    print("-" * 90)
    print("ANSWER:", response["answer"])
    print("-" * 90)
    print("RETRIEVED SOURCE CHUNKS:")
    for i, doc in enumerate(response["context"]):
        print(f"  Chunk {i+1} source:", doc.metadata.get("source", "Unknown"))
    print()

QUERY: What are the four waste categories and which colour container should each use?
------------------------------------------------------------------------------------------
ANSWER: I cannot answer based on the provided domain data.
------------------------------------------------------------------------------------------
RETRIEVED SOURCE CHUNKS:
  Chunk 1 source: my_data/02_city_waste_segregation_guide.txt
  Chunk 2 source: my_data/02_city_waste_segregation_guide.txt
  Chunk 3 source: my_data/02_city_waste_segregation_guide.txt

QUERY: What is the penalty for open burning of solid waste under RA 9003?
------------------------------------------------------------------------------------------
ANSWER: The penalty for open burning of solid waste under RA 9003 is a fine ranging from ₱300 to ₱1,000, or imprisonment of not less than — the exact imprisonment term is not specified in the provided context.
--------------------------------------------------------------------------------------

---
# Phase 5 — Practical Experiments and Deliverables

The three tasks required by the exercise are answered below, with their outputs recorded directly in this
notebook.

## Task 1 — Domain and Dataset Description

**Chosen domain:** Household solid-waste segregation and recycling in a Philippine city.

**Problem statement.** Residents are legally required to segregate their waste at source under Republic Act 9003,
but the rules are scattered across a national statute, local collection ordinances, and recycling guidance. The
result is predictable confusion: people do not know which bin a sachet belongs in, whether a greasy container can
be recycled, when hazardous items such as batteries may be handed over, or what penalty applies to open burning.
Waste is therefore mixed, collection is refused, and a large share of recoverable material ends up in landfill. A
retrieval-grounded assistant that answers **only** from the official documents gives residents a fast, correct,
and traceable answer — and, just as importantly, refuses to guess when the answer is not in the sources.

**SDG alignment.** This supports **SDG 11 (Sustainable Cities and Communities)**, whose Target 11.6 calls for
reducing the environmental impact of cities with special attention to municipal waste, and **SDG 12 (Responsible
Consumption and Production)**, whose Target 12.5 calls for substantially reducing waste generation through
prevention, reduction, recycling, and reuse. This individual domain was selected to sit close to my group's
SDG 11 and SDG 12 problem area, as instructed.

**Source files in `./my_data/`:**
1. `01_RA9003_solid_waste_law.txt` — plain-language summary of the Ecological Solid Waste Management Act of 2000:
   mandatory segregation at source, the Materials Recovery Facility requirement, the 25 % diversion target,
   prohibited acts, and penalties.
2. `02_city_waste_segregation_guide.txt` — practical household handbook: the four waste categories and their
   containers, a sample collection schedule, do's and don'ts, and home-composting guidance.
3. `03_recycling_circular_economy_faq.txt` — ten-question FAQ on the waste hierarchy, the circular economy,
   sachets, e-waste, Extended Producer Responsibility, food waste, and the difference between a sanitary landfill
   and an open dump.

The cell below prints the corpus statistics for the record.

In [ ]:
print("TASK 1 - DOMAIN AND DATASET DESCRIPTION")
print("=" * 70)
print("Domain           : Household solid-waste segregation and recycling (Philippines)")
print("SDG alignment    : SDG 11 (Sustainable Cities) and SDG 12 (Responsible Consumption)")
print(f"LLM              : {MODEL_NAME} via Groq")
print("Embedding model  : all-MiniLM-L6-v2 (384 dimensions)")
print("Vector database  : ChromaDB (in-memory)")
print("=" * 70)
print(f"Raw documents loaded : {len(raw_documents)}")
print(f"Total chunks indexed : {len(documents)}")
print(f"Chunk size / overlap : 500 / 50 characters")
print(f"Retriever top-k      : 3")
print("\nSource files in ./my_data/:")
for i, f in enumerate(sorted(os.listdir("my_data")), 1):
    size = os.path.getsize(os.path.join("my_data", f))
    print(f"  {i}. {f}  ({size:,} bytes)")

TASK 1 - DOMAIN AND DATASET DESCRIPTION
Domain           : Household solid-waste segregation and recycling (Philippines)
SDG alignment    : SDG 11 (Sustainable Cities) and SDG 12 (Responsible Consumption)
LLM              : openai/gpt-oss-20b via Groq
Embedding model  : all-MiniLM-L6-v2 (384 dimensions)
Vector database  : ChromaDB (in-memory)
Raw documents loaded : 3
Total chunks indexed : 38
Chunk size / overlap : 500 / 50 characters
Retriever top-k      : 3

Source files in ./my_data/:
  1. 01_RA9003_solid_waste_law.txt  (4,564 bytes)
  2. 02_city_waste_segregation_guide.txt  (4,198 bytes)
  3. 03_recycling_circular_economy_faq.txt  (4,272 bytes)


## Task 2 — Out-of-Domain Fallback Test

**Purpose.** To confirm that the grounding contract actually holds, the chatbot is asked a question with no
possible answer anywhere in the corpus. The three source documents concern waste segregation; they contain
nothing about space exploration.

**Out-of-domain query:** *"What is the average distance from Earth to Mars, and how long does a crewed mission
take to get there?"*

**Expected behaviour.** The retriever will still return three chunks, because a similarity search always returns
its nearest neighbours no matter how poor the match is. Those chunks will be irrelevant. The system prompt should
then cause the model to recognise that the context does not contain the answer and to reply with the exact
fallback sentence: *"I cannot answer based on the provided domain data."*

**Why this test matters.** This is the single most important safety behaviour of a domain chatbot. The base
model — `openai/gpt-oss-20b`, selected by Cell 4B — certainly "knows" the distance to Mars from its
pre-training, so an unconstrained model would answer happily. A correct refusal here proves that the prompt constraint is overriding the model's
own parametric memory — which is exactly the property that prevents confident, plausible, wrong answers in
production.

In [ ]:
out_of_domain_query = ("What is the average distance from Earth to Mars, "
                       "and how long does a crewed mission take to get there?")

response_ood = rag_chain.invoke({"input": out_of_domain_query})

print("TASK 2 - OUT-OF-DOMAIN FALLBACK TEST")
print("=" * 90)
print("CONFIGURATION : temperature = 0, fallback rule PRESENT")
print("QUERY         :", out_of_domain_query)
print("-" * 90)
print("ANSWER        :", response_ood["answer"])
print("-" * 90)
print("RETRIEVED CHUNKS (note that they are irrelevant to the question):")
for i, doc in enumerate(response_ood["context"]):
    src = doc.metadata.get("source", "Unknown")
    preview = doc.page_content[:110].replace("\n", " ")
    print(f"  Chunk {i+1} [{src}]: {preview}...")

TASK 2 - OUT-OF-DOMAIN FALLBACK TEST
CONFIGURATION : temperature = 0, fallback rule PRESENT
QUERY         : What is the average distance from Earth to Mars, and how long does a crewed mission take to get there?
------------------------------------------------------------------------------------------
ANSWER        : I cannot answer based on the provided domain data.
------------------------------------------------------------------------------------------
RETRIEVED CHUNKS (note that they are irrelevant to the question):
  Chunk 1 [my_data/02_city_waste_segregation_guide.txt]: weeks, when it becomes dark, crumbly, and earthy-smelling. The finished compost can be used for potted plants ...
  Chunk 2 [my_data/02_city_waste_segregation_guide.txt]: 2. SAMPLE COLLECTION SCHEDULE - Biodegradable waste: collected Monday, Wednesday, and Friday, from 6:00 AM to ...
  Chunk 3 [my_data/01_RA9003_solid_waste_law.txt]: one day but not more than fifteen days, or both. Heavier penalties, including lar

## Task 3 — Hallucination Stress-Test

**Purpose.** Task 2 showed the system behaving safely. This task deliberately removes both safeguards to measure
how much of that safety came from the configuration rather than from the model itself.

**The two changes (Step 1)**
1. **`temperature=0` → `temperature=1.0`.** Temperature scales the randomness of token sampling. At 0 the model
   always takes its most probable next token; at 1.0 it samples from the full probability distribution, so
   lower-probability — more inventive — continuations become reachable. This is the parameter most directly
   associated with creative drift.
2. **The fallback rule is deleted from the system prompt.** Without the sentence *"If the answer cannot be found
   in the context, reply: 'I cannot answer based on the provided domain data.'"*, the model is left with no
   sanctioned way to decline. An instruction-tuned model is trained to be helpful, so when refusal is not offered
   as an option it will tend to produce *something*.

**Line-by-line**
1. `llm_ungrounded = ChatGroq(..., temperature=1.0)` — a second model instance with sampling enabled.
2. `system_prompt_ungrounded` — the same role and context placeholder, **minus** the fallback sentence.
3. `ChatPromptTemplate.from_messages([...])` — rebuilds the template around the weakened instruction.
4. `create_stuff_documents_chain` / `create_retrieval_chain` — reassembles a parallel chain. Note that the
   **retriever is unchanged**: both configurations receive exactly the same three retrieved chunks, so any
   difference in the output is caused purely by temperature and the prompt, not by different evidence.
5. The same out-of-domain question from Task 2 is then re-asked (Step 2) and the two answers are printed
   side by side for comparison (Step 3).

**Why this is a fair experiment.** Only two variables change and the retrieval stage is held constant. That
isolation is what allows the difference in behaviour to be attributed to the safeguards themselves.

In [ ]:
# ---------- STEP 1: build the UNGROUNDED configuration ----------
llm_ungrounded = ChatGroq(
    model_name=MODEL_NAME,               # same model as the baseline
    temperature=1.0                      # CHANGED: sampling enabled
)

# CHANGED: the fallback instruction line has been deleted
system_prompt_ungrounded = (
    "You are a specialized AI assistant for the user's uploaded domain.\n"
    "Answer questions strictly using ONLY the provided context below.\n\n"
    "Context:\n{context}"
)

prompt_ungrounded = ChatPromptTemplate.from_messages([
    ("system", system_prompt_ungrounded),
    ("human", "{input}"),
])

# The SAME retriever is reused, so both chains see identical evidence
combine_docs_chain_ungrounded = create_stuff_documents_chain(llm_ungrounded, prompt_ungrounded)
rag_chain_ungrounded = create_retrieval_chain(retriever, combine_docs_chain_ungrounded)

# ---------- STEP 2: re-ask the exact same out-of-domain question ----------
response_ungrounded = rag_chain_ungrounded.invoke({"input": out_of_domain_query})

# ---------- STEP 3: side-by-side comparison ----------
print("TASK 3 - HALLUCINATION STRESS-TEST")
print("=" * 90)
print("QUERY (identical in both runs):", out_of_domain_query)
print("=" * 90)
print("\n[A] BASELINE  - temperature = 0, fallback rule PRESENT")
print("-" * 90)
print(response_ood["answer"])
print("\n" + "=" * 90)
print("\n[B] UNGROUNDED - temperature = 1.0, fallback rule REMOVED")
print("-" * 90)
print(response_ungrounded["answer"])
print("\n" + "=" * 90)
print("Retrieved chunks were identical in both runs (same retriever, k = 3).")

TASK 3 - HALLUCINATION STRESS-TEST
QUERY (identical in both runs): What is the average distance from Earth to Mars, and how long does a crewed mission take to get there?

[A] BASELINE  - temperature = 0, fallback rule PRESENT
------------------------------------------------------------------------------------------
I cannot answer based on the provided domain data.


[B] UNGROUNDED - temperature = 1.0, fallback rule REMOVED
------------------------------------------------------------------------------------------
I’m sorry, but the information you’re asking for isn’t included in the context that was provided.

Retrieved chunks were identical in both runs (same retriever, k = 3).


### Task 3 — Repeatability Check

A single sample at `temperature=1.0` could be luck. Because sampling is now stochastic, the same prompt can give a
different answer each time. This cell asks the identical question three more times under the ungrounded
configuration to show whether the behaviour is consistent or variable — variability across runs is itself
evidence of reduced reliability.

In [ ]:
print("UNGROUNDED CONFIGURATION - THREE REPEAT RUNS (temperature = 1.0, no fallback rule)")
print("=" * 90)
for run in range(1, 4):
    r = rag_chain_ungrounded.invoke({"input": out_of_domain_query})
    print(f"\nRun {run}:")
    print("-" * 90)
    print(r["answer"])
print("\n" + "=" * 90)
print("Compare the three answers above. Differences between runs demonstrate that")
print("temperature = 1.0 makes the system non-deterministic and therefore less reliable.")

UNGROUNDED CONFIGURATION - THREE REPEAT RUNS (temperature = 1.0, no fallback rule)

Run 1:
------------------------------------------------------------------------------------------
I’m sorry, but I can’t provide that information.

Run 2:
------------------------------------------------------------------------------------------
I’m sorry, but the provided context does not include any information about the average distance from Earth to Mars or the travel time for a crewed mission.

Run 3:
------------------------------------------------------------------------------------------
I’m sorry, but I don’t have that information.

Compare the three answers above. Differences between runs demonstrate that
temperature = 1.0 makes the system non-deterministic and therefore less reliable.


### Task 3 — Observations and Discussion

*Recorded from the printed outputs of Cells 7, 23, 25 and 27 in this run. These are what actually happened, not
what was expected.*

**Did the baseline configuration refuse correctly?**  
Yes. On the out-of-domain Mars query the baseline (`temperature = 0`, fallback rule present) returned the exact
canonical sentence *"I cannot answer based on the provided domain data."* — character for character the string
specified in the system prompt. The retriever had still returned three chunks, as a similarity search always
does, but they were about home composting, the collection schedule and RA 9003 penalties, and the model
correctly recognised that none of them answered the question.

**What did the ungrounded configuration produce?**  
It refused as well — which was *not* the expected result. Removing the fallback rule and raising the temperature
to 1.0 did not cause the model to answer from pre-training, and it did not invent a distance. Across all four
ungrounded generations it produced zero hallucinated facts. The honest reading is not that the prompt
engineering was pointless, but that the retrieved context was itself a strong enough signal for an
instruction-tuned model to decline. Modern instruction-tuned models have internalised part of the grounding
behaviour that the fallback rule was written to impose.

**Were the repeat runs identical or different?**  
All four were different — no two alike:

| Generation | Response |
|---|---|
| Task 3 [B] | "I'm sorry, but the information you're asking for isn't included in the context that was provided." |
| Repeat 1 | "I'm sorry, but I can't provide that information." |
| Repeat 2 | "I'm sorry, but the provided context does not include any information about the average distance from Earth to Mars or the travel time for a crewed mission." |
| Repeat 3 | "I'm sorry, but I don't have that information." |

The retrieved chunks were byte-identical in every generation, so the prompt was the only variable. Note the
spread: Repeat 1 is an eight-word refusal that says nothing about the subject, while Repeat 2 restates the entire
question back. Both are refusals, but no string match, prefix test or length heuristic catches both. The
baseline, by contrast, returns one fixed sentence every time.

**Which safeguard appears to matter more — the temperature setting or the fallback rule?**  
On this evidence, neither prevented a hallucination, because none occurred. What they jointly delivered was
**determinism**. That is not a cosmetic difference: a canonical refusal string can be detected by an exact string
match and routed to a human operator or a logged unanswered-question queue, whereas four wordings that vary from
eight words to a full restatement of the question cannot be detected reliably at all. `temperature = 0` makes
decoding greedy and the output reproducible; the fallback rule fixes *which* sentence that output is. The
measured value of both safeguards here is machine-parseability, not hallucination suppression.

**One further finding, from Cell 7.**  
The in-domain results were not perfect. Query 1 ("the four waste categories and their container colours")
returned the fallback string even though the answer *is* in `02_city_waste_segregation_guide.txt`, and all three
retrieved chunks came from that very file. The category list runs to roughly 1,400 characters, so a 500-character
chunk size spreads it across three chunks, none of which individually matches a question phrased around "four
categories" and "colour" — and with `k = 3` the composting and schedule passages crowded them out. Query 2 failed
the same way in milder form: it gave the fine correctly as ₱300 to ₱1,000 but had to report that "the exact
imprisonment term is not specified in the provided context". That term — *"one day but not more than fifteen
days"* — is demonstrably in the index: it was returned as a retrieved chunk during the Task 2 out-of-domain test,
for a question that had nothing to do with penalties. It simply was not in the top three for the query that
actually needed it.

This is a **false refusal**, and it is the cost side of a strict grounding contract: the contract cannot
distinguish *"the corpus does not contain this"* from *"retrieval did not surface it"*, so a retrieval miss
becomes a confident refusal. Both failures are retrieval-side, not generation-side, and both are fixable without
touching the prompt — raise `k` to 5, increase the chunk size to about 1,000 characters so enumerated lists stay
intact, or switch the retriever to maximal marginal relevance.

**A note on index integrity.**  
An earlier session re-ran the indexing cell without restarting the runtime. `Chroma.from_documents()` appends to
an existing collection rather than replacing it, so the store silently held two copies of every chunk: a top-3
search then returned the same passage twice plus one other, the effective `k` fell from 3 to about 2, and the
RA 9003 penalties passage dropped out of the Task 2 retrieval entirely. Nothing errored — the answers just
quietly got worse. Cell 4 now drops any leftover collection before indexing and prints an explicit integrity
check, which reads `OK - no duplicates` in this run.

These notes became Section III of the IEEE lab report.


---
## Summary

This notebook implemented a full RAG pipeline over a private, domain-specific corpus on household waste
segregation and recycling, then tested it under two deliberately different configurations.

**Architecture:** three `.txt` documents (13,034 bytes) → `RecursiveCharacterTextSplitter` (500 / 50) → 38 chunks
→ `all-MiniLM-L6-v2` 384-dimensional embeddings → in-memory ChromaDB (integrity-checked at 38 vectors) → top-3
similarity retriever → grounded system prompt → Groq `openai/gpt-oss-20b`.

**Safeguards evaluated:** deterministic decoding (`temperature = 0`), an explicit instruction to answer only from
retrieved context, an explicit permitted refusal string, and runtime-only handling of the API key via `getpass`.

**Two deviations from the lab template, both forced and both documented.** `langchain.chains` no longer exists in
LangChain 1.x, so every langchain package is pinned below version 1.0 (Cell 1). The template's model
`llama-3.1-8b-instant` returns `404 model_not_found` because Groq deprecated it, so Cell 4B queries the Groq
`/v1/models` endpoint at run time and selects an available model from a ranked preference list — resolving to
`openai/gpt-oss-20b` out of the 14 offered, rather than hard-coding a replacement that could itself expire.

**Results.** Zero hallucinated facts across four in-domain queries and four ungrounded out-of-domain
generations. Two findings are less obvious than that headline:

1. **Over-refusal is real, measurable and reproducible.** One of the four in-domain queries was refused even
   though its answer is in the corpus, because a 500-character chunk size split the relevant list across three
   chunks and `k = 3` did not reach them. The same refusal occurred in every session tested. That is a 1-in-4
   false-refusal rate on this small query set — a genuine failure mode, and a cost of the grounding contract
   that a zero-hallucination figure alone would hide.
2. **The fallback rule bought determinism, not safety.** The ungrounded configuration (`temperature = 1.0`, no
   fallback rule) refused every time as well, but in four different wordings across four generations, ranging
   from an eight-word refusal to a full restatement of the question. What the rule actually guarantees is one
   exact, machine-parseable refusal string.

**Deliverables:** this executed notebook, and a two-page IEEE-formatted lab report covering the architecture, the
parameter table, the test-query results and the hallucination analysis.
